# Notebook 02 — Camada Bronze (Ingestão)

## Arquitetura Medallion

A arquitetura Medallion organiza os dados em três camadas progressivas de qualidade:

- **Bronze**: Ingestão bruta (raw) — dados como vieram da fonte, preservando o formato original
- **Silver**: Dados limpos, padronizados e deduplicados, prontos para análise
- **Gold**: Modelo dimensional (Star Schema) otimizado para consumo por ferramentas de BI

Neste notebook, realizamos a **ingestão dos arquivos Parquet** gerados no NB01 para a camada Bronze, armazenando-os em tabelas Delta Lake.


## 1. Criação da SparkSession

Iniciamos uma SparkSession com suporte a Delta Lake. O Delta Lake é uma camada de armazenamento que adiciona transações ACID, versionamento e schema evolution sobre arquivos Parquet.


In [ ]:
import os
from pyspark.sql import SparkSession

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
BRONZE_DIR = os.path.join(DATA_DIR, "bronze")
BRONZE_DELTA_DIR = os.path.join(DATA_DIR, "bronze_delta")

spark = (
    SparkSession.builder
    .appName("NB02_Bronze")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession iniciada. Versão: {spark.version}")
print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Diretório Bronze (fonte): {BRONZE_DIR}")
print(f"Diretório Delta (destino): {BRONZE_DELTA_DIR}")


## 2. Leitura dos Arquivos Fonte (Parquet)

Lemos cada arquivo Parquet gerado no NB01 e inspecionamos seu schema e contagem de linhas.


In [ ]:
clientes_path = os.path.join(BRONZE_DIR, "clientes.parquet")
df_clientes_raw = spark.read.parquet(clientes_path)
print("Schema — clientes:")
df_clientes_raw.printSchema()
print(f"Registros: {df_clientes_raw.count()}")
df_clientes_raw.show(5, truncate=False)


In [ ]:
produtos_path = os.path.join(BRONZE_DIR, "produtos.parquet")
df_produtos_raw = spark.read.parquet(produtos_path)
print("Schema — produtos:")
df_produtos_raw.printSchema()
print(f"Registros: {df_produtos_raw.count()}")
df_produtos_raw.show(5, truncate=False)


In [ ]:
pedidos_path = os.path.join(BRONZE_DIR, "pedidos.parquet")
df_pedidos_raw = spark.read.parquet(pedidos_path)
print("Schema — pedidos:")
df_pedidos_raw.printSchema()
print(f"Registros: {df_pedidos_raw.count()}")
df_pedidos_raw.show(5, truncate=False)


## 3. Escrita em Tabelas Delta (Camada Bronze)

Cada DataFrame é salvo como tabela Delta. Utilizamos `mode("overwrite")` para garantir a recriação completa e habilitamos `mergeSchema` para evolução futura de schema.


In [ ]:
bronze_clientes_path = os.path.join(BRONZE_DELTA_DIR, "bronze_clientes")
df_clientes_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(bronze_clientes_path)
print(f"[OK] bronze_clientes salvo em {bronze_clientes_path}")


In [ ]:
bronze_produtos_path = os.path.join(BRONZE_DELTA_DIR, "bronze_produtos")
df_produtos_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(bronze_produtos_path)
print(f"[OK] bronze_produtos salvo em {bronze_produtos_path}")


In [ ]:
bronze_pedidos_path = os.path.join(BRONZE_DELTA_DIR, "bronze_pedidos")
df_pedidos_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(bronze_pedidos_path)
print(f"[OK] bronze_pedidos salvo em {bronze_pedidos_path}")


## 4. Validação da Camada Bronze

Relemos as tabelas Delta e verificamos se a contagem de registros confere com os arquivos fonte.


In [ ]:
df_bronze_clientes = spark.read.format("delta").load(bronze_clientes_path)
df_bronze_produtos = spark.read.format("delta").load(bronze_produtos_path)
df_bronze_pedidos = spark.read.format("delta").load(bronze_pedidos_path)

count_raw_clientes = df_clientes_raw.count()
count_raw_produtos = df_produtos_raw.count()
count_raw_pedidos = df_pedidos_raw.count()

count_bronze_clientes = df_bronze_clientes.count()
count_bronze_produtos = df_bronze_produtos.count()
count_bronze_pedidos = df_bronze_pedidos.count()

print("=" * 60)
print("VALIDAÇÃO DE CONTAGEM — CAMADA BRONZE")
print("=" * 60)
print(f"{'Tabela':<20} {'Fonte (Parquet)':<18} {'Bronze (Delta)':<18} {'Status':<10}")
print("-" * 66)
status_c = "OK" if count_raw_clientes == count_bronze_clientes else "ERRO"
status_p = "OK" if count_raw_produtos == count_bronze_produtos else "ERRO"
status_pe = "OK" if count_raw_pedidos == count_bronze_pedidos else "ERRO"
print(f"{'clientes':<20} {count_raw_clientes:<18} {count_bronze_clientes:<18} {status_c:<10}")
print(f"{'produtos':<20} {count_raw_produtos:<18} {count_bronze_produtos:<18} {status_p:<10}")
print(f"{'pedidos':<20} {count_raw_pedidos:<18} {count_bronze_pedidos:<18} {status_pe:<10}")


## 5. Encerramento da SparkSession

Liberamos os recursos do Spark.


In [ ]:
spark.stop()
print("SparkSession encerrada.")


## Resumo da Camada Bronze

| Tabela | Formato Origem | Formato Destino | Registros |
|--------|---------------|-----------------|-----------|
| clientes | Parquet | Delta Lake | 1.500 |
| produtos | Parquet | Delta Lake | 250 |
| pedidos | Parquet | Delta Lake | 8.000 |

Os dados brutos foram ingeridos com sucesso na camada Bronze. Nenhuma transformação foi aplicada — preservamos os dados exatamente como recebidos.

No próximo notebook (**NB03**), aplicaremos transformações de limpeza e padronização na **Camada Silver**.
